# Student Social Media & Mental Health — Complete ML & Explainable AI Notebook
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sudharsan0503/MLProject-mentalhealth/blob/main/mental_health.ipynb)

> **SYNTHETIC DATA DISCLAIMER**: The dataset (`Student_Mental_Health_Extended.csv`, 5,400 rows, 33 columns) is **SYNTHETIC**. All findings, feature attributions, and predictions are for educational and machine learning demonstration purposes only and do **not** represent clinical or medical assessments.

## 0. Google Colab Environment Setup
Run the cell below first. If you opened this notebook in **Google Colab**, it will automatically clone the GitHub repository and install the required libraries (`shap`, `lime`, `xgboost`, `lightgbm`, `optuna`, `reportlab`, `seaborn`).

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('MLProject-mentalhealth'):
        !git clone https://github.com/sudharsan0503/MLProject-mentalhealth.git
    if os.path.basename(os.getcwd()) != 'MLProject-mentalhealth':
        os.chdir('MLProject-mentalhealth')
    !pip install -q shap lime xgboost lightgbm optuna reportlab seaborn fastapi uvicorn httpx pytest
sys.path.insert(0, os.path.abspath('.'))
print('Working directory:', os.getcwd())

## Phase 1: Data Understanding & Leak-Free Pre-Processing
- Loads `data/Student_Mental_Health_Extended.csv` (5,400 rows, 33 columns).
- Drops `Student_ID` (identifier) and `Mental_Health_Category` (target leakage when predicting `Mental_Health_Score`).
- Engineers 5 domain features inside `FeatureEngineer` (`Sleep_Deficit`, `Night_Screen_Ratio`, `Unlocks_Per_Hour`, `Digital_Overload`, `Wellbeing_Buffer`).
- Applies median/mode imputation, `OrdinalEncoder` for ordered categoricals, `OneHotEncoder(handle_unknown='ignore')` for nominals, and `StandardScaler` inside a leak-free `Pipeline`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import build_preprocessor, FeatureEngineer

df = pd.read_csv('data/Student_Mental_Health_Extended.csv')
print('Shape:', df.shape)
print('\nMissing Values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nDuplicates:', df.duplicated().sum())
df.head()

## Phase 2: Exploratory Data Analysis & 5-Method Feature Importance
Computes and compares all 5 feature importance methods:
1. Pearson & Spearman correlation + Mutual Information
2. Tree-based impurity importance (Random Forest & Gradient Boosting)
3. Permutation importance on the test set (`n_repeats=20`)
4. Global SHAP mean `|value|`
5. Domain-grouped importance (Psychosocial, Digital, Lifestyle, Demographic)

In [ ]:
import importlib
from IPython.display import Image, display

eda_mod = importlib.import_module('src.01_data_understanding_and_eda')
df, fi_df = eda_mod.run_eda_and_feature_importance()
display(Image('reports/feature_importance.png', width=800))
fi_df.head(15)

## Phase 3: 16-Model Comparative Training & Automated Final Verdict
Trains and evaluates Linear Regression, Ridge, Lasso, ElasticNet, KNN, SVR, Decision Tree, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, XGBoost, LightGBM, MLPRegressor, Voting Ensemble, and Stacking Ensemble using 5-fold Cross-Validation.

In [ ]:
from src.train import train_and_evaluate_all
from IPython.display import Image, display

results_df, best_pipeline = train_and_evaluate_all()
display(Image('reports/model_comparison.png', width=850))
results_df

## Phase 4: Explainable AI (SHAP, LIME, PDP/ICE & Counterfactual What-If)
Generates SHAP summary beeswarm/bar plots, LIME explanations for Low/Medium/High score students, Partial Dependence + ICE curves, and counterfactual action plans.

In [ ]:
from src.explain import run_explainability, find_counterfactual
from IPython.display import Image, display

run_explainability()
display(Image('reports/shap_plots/shap_summary_beeswarm.png', width=750))
display(Image('reports/pdp_ice_plots.png', width=850))

## Phase 5 & 6: Run Automated Pytest Suite & Launch Interactive Web UI in Colab
Run the cells below to execute the `pytest` suite or open the interactive Web UI inside Google Colab.

In [ ]:
!pytest tests/

In [ ]:
# Optional: Launch the interactive Web UI directly inside Google Colab
import sys, subprocess, time
if 'google.colab' in sys.modules:
    from google.colab import output
    proc = subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])
    time.sleep(3)
    output.serve_kernel_port_as_iframe(8000, height='850')
else:
    print('Run locally via: uvicorn app.main:app --port 8000')